# SkillSync v2 — Word2Vec + NER Skill Clustering

**Dataset:** LinkedIn Job Postings 2023–2024 (Kaggle)
Link: https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

**File used:** `postings.csv`
**Column used:** `description` (raw unstructured job posting text)

**Goal:** Instead of a hardcoded skill list, automatically:
1. Extract candidate skill/tool entities from raw text using **NER (spaCy)**
2. Train **Word2Vec** on the same job description corpus
3. **Cluster** similar/duplicate skill mentions together (e.g. "ML" and "Machine Learning")
4. Produce a clean, auto-generated skill vocabulary

> Upload `postings.csv` to the Colab runtime before running Cell 3.


In [1]:
!pip install -qU gensim spacy scikit-learn
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 4.2 MB/s eta 0:00:03
     ---- ----------------------------------- 1.3/12.8 MB 3.9 MB/s eta 0:00:03
     ------ --------------------------------- 2.1/12.8 MB 3.8 MB/s eta 0:00:03
     --------- ------------------------------ 3.1/12.8 MB 4.0 MB/s eta 0:00:03
     ---------- ----------------------------- 3.4/12.8 MB 4.0 MB/s eta 0:00:03
     ----------- ---------------------------- 3.7/12.8 MB 3.2 MB/s eta 0:00:03
     ------------- -------------------------- 4.2/12.8 MB 3.1 MB/s eta 0:00:03
     -------------- ------------------------- 4.7/12.8 MB 2.8 MB/s eta 0:00:03
     ---------------- ----------------------- 5.2/12.8 MB 2.8 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 2.8 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 2.8 MB/s eta 0:00:03
     -------------------- ------------------- 6.6/12.8 MB 2

In [7]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.1 MB/s eta 0:00:11
     - -------------------------------------- 0.5/12.8 MB 1.1 MB/s eta 0:00:11
     -- ------------------------------------ 0.8/12.8 MB 838.9 kB/s eta 0:00:15
     --- ----------------------------------- 1.0/12.8 MB 949.8 kB/s eta 0:00:13
     ---- ----------------------------------- 1.3/12.8 MB 1.0 MB/s eta 0:00:12
     ---- ----------------------------------- 1.6/12.8 MB 1.0 MB/s eta 0:00:11
     ---- ----------------------------------- 1.6/12.8 MB 1.0 MB/s eta 0:00:11
     ----- ---------------------------------- 1.8/12.8 MB 1.0 MB/s eta 0:00:11
     ------- -------------------------------- 2.4/12.8 MB 1.1 MB/s eta 0:00:10
     --------- ------------------------------ 2.9/12.8 MB 1.3 MB/s eta 

In [ ]:
!pip install spacy
print(spacy.__version__)

In [14]:
!pip install gensim

In [17]:
!pip install spacy

In [23]:
!pip install spacy

In [24]:
import sys
print(sys.executable)

C:\Users\ch.sindhu sri\AppData\Local\Programs\Python\Python313\python.exe


In [1]:
import numpy
print(numpy.__version__)

2.4.6


In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import spacy
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Load dataset

In [2]:
df = pd.read_csv("postings.csv")
df = df[['job_id', 'description']]

## 2. Clean nulls/duplicates

In [3]:
df.isnull().sum()

job_id         0
description    7
dtype: int64

In [7]:
df.dropna(inplace=True)
df.drop_duplicates(subset='description', inplace=True)
df.reset_index(drop=True, inplace=True)

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9+#. ]", " ", text)   # keep + # . for C++, C#, .NET
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [9]:
nlp = spacy.load("en_core_web_sm")

In [11]:
def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text.strip().lower() for ent in doc.ents
                if ent.glabel_ in ("ORG", "PRODUCT", "GPE", "LANGUAGE")]
    return entities

sample_df = df.sample(3000, random_state=42).reset_index(drop=True)
sample_df['entities'] = sample_df['description'].apply(extract_entities)

In [12]:
all_entities = [ent for row in sample_df['entities'] for ent in row]
print("Total candidates extracted:", len(all_entities))
print("Unique candidates:", len(set(all_entities)))

Total candidates extracted: 47630
Unique candidates: 20856


In [ ]:
with open("ner_candidates.pkl", "wb") as f:
    pickle.dump(all_entities, f)

In [ ]:
def tokenize(text):
    return clean_text(text).split()

corpus = [tokenize(desc) for desc in df['description']]

In [ ]:
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,          # skip-gram (better for smaller/rarer skill words)
    epochs=10
)

w2v_model.save("skillsync_word2vec.model")
print("Vocabulary size:", len(w2v_model.wv))

In [ ]:
unique_entities = list(set(all_entities))

valid_entities = []
entity_vectors = []

for ent in unique_entities:
    words = ent.split()
    vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if vecs:
        valid_entities.append(ent)
        entity_vectors.append(np.mean(vecs, axis=0))

entity_vectors = np.array(entity_vectors)
print("Entities with vectors:", entity_vectors.shape)

In [ ]:
scores = []
k_range = range(10, 60, 5)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(entity_vectors)
    scores.append(silhouette_score(entity_vectors, labels))

best_k = list(k_range)[np.argmax(scores)]
print("Best k:", best_k)

In [ ]:
cluster_df = pd.DataFrame({
    "entity": valid_entities,
    "cluster": cluster_labels
})

for c in sorted(cluster_df['cluster'].unique())[:10]:
    print(f"Cluster {c}:", cluster_df[cluster_df['cluster'] == c]['entity'].tolist())

In [ ]:
cluster_counts = cluster_df.groupby('cluster')['entity'].apply(list)

canonical_skill_map = {}
for cluster_id, entities in cluster_counts.items():
    canonical = min(entities, key=len)
    canonical_skill_map[cluster_id] = canonical

cluster_df['canonical_skill'] = cluster_df['cluster'].map(canonical_skill_map)
cluster_df.to_csv("auto_extracted_skills.csv", index=False)

## 15. Save final clean skill vocabulary (replaces the hardcoded list)

In [ ]:
final_skill_list = sorted(cluster_df['canonical_skill'].unique())
pickle.dump(final_skill_list, open("final_skill_list.pkl", "wb"))

print("Final auto-discovered skill count:", len(final_skill_list))
print(final_skill_list[:30])